Didnt get output tried for croping


In [1]:
import os
import shutil
import random

def split_dataset(source_dir, dest_dir, train_ratio=0.7, val_ratio=0.15):
    classes = ['real', 'fake']
    for cls in classes:
        files = os.listdir(os.path.join(source_dir, cls))
        random.shuffle(files)

        total = len(files)
        train_end = int(train_ratio * total)
        val_end = int((train_ratio + val_ratio) * total)

        splits = {
            'train': files[:train_end],
            'val': files[train_end:val_end],
            'test': files[val_end:]
        }

        for split, split_files in splits.items():
            split_dir = os.path.join(dest_dir, split, cls)
            os.makedirs(split_dir, exist_ok=True)

            for file in split_files:
                src = os.path.join(source_dir, cls, file)
                dst = os.path.join(split_dir, file)
                shutil.copy(src, dst)

    print("✅ Dataset split into train/val/test")

# Example usage
split_dataset("cropped_faces", "dataset")


✅ Dataset split into train/val/test


In [2]:
import os

print("REAL:", len(os.listdir("cropped_faces/real")))
print("FAKE:", len(os.listdir("cropped_faces/fake")))


REAL: 0
FAKE: 0


In [3]:
import os
from mtcnn import MTCNN
import cv2

detector = MTCNN()
video_path = "Dataset/real/your_real_video.mp4"  # replace with actual video path
save_dir = "cropped_faces"
label = "real"

cap = cv2.VideoCapture(video_path)
count = 0
frame_num = 0

while cap.isOpened() and count < 5:
    ret, frame = cap.read()
    if not ret:
        break

    frame_num += 1
    if frame_num % 5 != 0:
        continue

    faces = detector.detect_faces(frame)
    print(f"Frame {frame_num}: Found {len(faces)} face(s)")

    for i, face in enumerate(faces):
        x, y, w, h = face['box']
        x, y = max(0, x), max(0, y)
        cropped_face = frame[y:y+h, x:x+w]
        resized_face = cv2.resize(cropped_face, (160, 160))

        out_dir = os.path.join(save_dir, label)
        os.makedirs(out_dir, exist_ok=True)
        file_path = os.path.join(out_dir, f"{label}_{count}.jpg")
        cv2.imwrite(file_path, resized_face)
        print(f"✅ Saved {file_path}")
        count += 1
        break

cap.release()


C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please 

In [1]:
import os
import cv2
from mtcnn import MTCNN
from tqdm import tqdm

def crop_faces(input_folder, output_folder):
    # Create output directory structure
    for label in ['real', 'fake']:
        os.makedirs(os.path.join(output_folder, label), exist_ok=True)
    
    detector = MTCNN()
    
    for label in ['real', 'fake']:
        input_path = os.path.join(input_folder, label)
        output_path = os.path.join(output_folder, label)
        
        # Verify input directory exists
        if not os.path.exists(input_path):
            print(f"⚠️ Directory not found: {input_path}")
            continue
            
        # Get image files with multiple extensions
        image_files = []
        for ext in ['.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.JPEG']:
            image_files.extend([f for f in os.listdir(input_path) if f.lower().endswith(ext)])
            
        if not image_files:
            print(f"⚠️ No images found in {input_path}")
            continue
            
        print(f"\nProcessing {len(image_files)} {label} images...")
        
        for filename in tqdm(image_files, desc=label):
            img_path = os.path.join(input_path, filename)
            output_file = os.path.join(output_path, filename)
            
            try:
                # Read image
                img = cv2.imread(img_path)
                if img is None:
                    print(f"\n❌ Failed to read {filename}")
                    continue
                
                # Convert to RGB
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                # Detect faces
                faces = detector.detect_faces(img_rgb)
                if not faces:
                    continue
                
                # Crop primary face
                x, y, w, h = faces[0]['box']
                x, y = max(0, x), max(0, y)
                face = img_rgb[y:y+h, x:x+w]
                
                # Resize and save
                face = cv2.resize(face, (224, 224))
                cv2.imwrite(output_file, cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
                
            except Exception as e:
                print(f"\n⚠️ Error processing {filename}: {str(e)}")
                continue

    print("\n✅ Processing complete")

# Example usage
crop_faces("frames", "cropped_faces")

C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\bhara\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please 

⚠️ No images found in frames\real
⚠️ No images found in frames\fake

✅ Processing complete
